# AfriVoices — SOUMISSION FENÊTRES **v2** (recollage des logits)

Remplace `afrivoices_soumission_fenetres.ipynb`. Quatre changements, tous validés (ou
mesurables) dans le notebook P0 (`afrivoices_p0_devlong_padding_fenetres.ipynb`) :

1. **Recollage des LOGITS, pas du texte** : les logits des fenêtres sont concaténés sur la
   ligne de temps du clip (chaque chevauchement tranché en son milieu, sur la coupe-silence) puis décodés en **une seule
   passe LM** par clip → le KenLM voit tout le contexte ; plus de duplication quand le
   recollage textuel échouait, plus de fragments de mots aux frontières.
2. **Coupes au silence** : la frontière de chaque fenêtre est placée au minimum d'énergie
   RMS dans les 3 dernières secondes de la fenêtre cible (au lieu de 15.0s pile) → on ne
   coupe plus de mots en deux. Fenêtres ≤ 16s, toujours dans le régime < 20s de l'entraînement.
3. **Dernière fenêtre alignée sur la fin** (`[len-15s, len]`) : plus jamais de fenêtre de
   2s décodée seule (source d'hallucinations LM).
4. **Troncature anti-padding** : chaque logit est tronqué à sa vraie longueur avant décodage
   (les frames de padding n'ont jamais été entraînées — bug mesuré en §5 du notebook P0,
   il touchait TOUTES les soumissions, fenêtres ou pas).

Bonus : **(α, β) par langue** (§2, dict `AB`) et **beam élargi optionnel** pour kln/mas
(§2, `BEAM_LANG`) — reporte les valeurs de la grille §8 du notebook P0 si elles gagnent.

**Mode d'emploi** : §1 (install + REDÉMARRER) → §2 (config) → §3 (rechargement) →
§4 (test) → §5 (inférence, reprise auto tous les 10 parquets) → §6 (asserts + CSV).
Compare au leaderboard : **0.40283** (direct) et ta soumission fenêtres v1.
Conformité edge inchangée : même modèle, même LM ; le fenêtrage ajoute ~15 % de calcul
sur les clips longs (chevauchements) → RTF largement ≤ 2 ; RAM identique.

## 1. Dépendances (redémarrer le runtime après la 1ère exécution)

In [ ]:
import importlib, subprocess
need = [m for m in ["kenlm", "pyctcdecode"] if importlib.util.find_spec(m) is None]
if need:
    print("installation de", need, "...")
    subprocess.run(["pip", "install", "-q", "pyctcdecode"], check=False)
    subprocess.run(["pip", "install", "-q", "https://github.com/kpu/kenlm/archive/master.zip"], check=False)
    print("⚠️ REDÉMARRE le runtime (Exécution > Redémarrer la session), puis relance cette cellule.")
else:
    print("✓ kenlm + pyctcdecode disponibles — continue en §2")

## 2. CONFIG — le seul bloc à modifier

In [ ]:
# ============================================================
MODEL_NAME = "baobab-asr-v9-2"      # modèle à soumettre (dossier sur le Drive)
LM_SUBDIR  = "lm_v2"
TAG = "v9_2_fenetres_v2"

# (alpha, beta) PAR LANGUE — défaut : le réglage global validé (0.7, 0.5).
# Remplace UNIQUEMENT les langues marquées "ADOPTER" par la grille §8 du notebook P0.
AB = {"swa": (0.7, 0.5), "kik": (0.7, 0.5), "luo": (0.7, 0.5),
      "kln": (0.7, 0.5), "mas": (0.7, 0.5), "som": (0.7, 0.5)}

# beam élargi par langue (optionnel — seulement si le test §8 du notebook P0 gagne >0.01)
BEAM_LANG = {}                      # ex: {"kln": {"beam_width": 256, "token_min_logp": -7.0}}

# unigrams du décodeur (optionnel — ex: {"swa": 200000} si mesuré gagnant)
UNIGRAM_TOP = 50000
UNIGRAM_TOP_LANG = {}

# fenêtrage
WIN_SEC, OVER_SEC, SEUIL_SEC, SEARCH_SEC = 15.0, 2.0, 20.0, 3.0
# ============================================================

from google.colab import drive
drive.mount("/content/drive")
import os
BASE  = "/content/drive/MyDrive/afrivoices"
MODEL = f"{BASE}/{MODEL_NAME}"
LM    = f"{BASE}/{LM_SUBDIR}"
PARTIEL = f"{BASE}/submission_{TAG}_partiel.csv"
FINAL   = f"{BASE}/submission_{TAG}.csv"
assert os.path.isdir(MODEL), f"modèle absent: {MODEL}"
assert os.path.isdir(LM), f"LM absent: {LM}"
print(f"modèle: {MODEL_NAME} | LM: {LM_SUBDIR}")
for l, (a, b) in AB.items(): print(f"  {l}: alpha={a} beta={b}" + (f" beam={BEAM_LANG[l]}" if l in BEAM_LANG else ""))
print(f"sortie: {FINAL}")

## 3. Rechargement : modèle + labels + décodeurs (α/β par langue) + decode_robuste

In [ ]:
import torch, io, base64, numpy as np, soundfile as sf, librosa, tempfile
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "" if device == "cuda" else "⚠️ GPU recommandé (45-70 min vs plusieurs heures)")

model = Wav2Vec2BertForCTC.from_pretrained(MODEL, local_files_only=True).to(device).eval()
processor = Wav2Vec2BertProcessor.from_pretrained(MODEL, local_files_only=True)

raw = [t for t, _ in sorted(processor.tokenizer.get_vocab().items(), key=lambda kv: kv[1])]
blank_tok = processor.tokenizer.pad_token
labels, n = [], 0
for t in raw:
    if t == blank_tok: labels.append("")
    elif t in ("|", " "): labels.append(" ")
    elif t in {"[UNK]", "<s>", "</s>"}: n += 1; labels.append("\u2047" * n)
    else: labels.append(t)
assert labels.count("") == 1 and labels.count(" ") == 1

def decode_robuste(b):
    if not b: raise ValueError("vide")
    try: arr, sr = sf.read(io.BytesIO(b), dtype="float32")
    except Exception:
        try: arr, sr = sf.read(io.BytesIO(base64.b64decode(b)), dtype="float32")
        except Exception:
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tf:
                try: tf.write(base64.b64decode(b))
                except Exception: tf.write(b)
                p = tf.name
            arr, sr = librosa.load(p, sr=16000, mono=True); os.unlink(p)
            return arr.astype(np.float32)
    if arr.ndim > 1: arr = arr.mean(axis=1)
    if sr != 16000: arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    return arr.astype(np.float32)

def unigrams(lang, top):
    c = Counter()
    with open(f"{LM}/{lang}.txt", encoding="utf-8") as f:
        for line in f: c.update(line.split())
    return [w for w, _ in c.most_common(top)]

decoders, decode_kwargs = {}, {}
for lang in ["swa", "kik", "luo", "kln", "mas", "som"]:
    a, b = AB[lang]
    top = UNIGRAM_TOP_LANG.get(lang, UNIGRAM_TOP)
    decoders[lang] = build_ctcdecoder(labels, kenlm_model_path=f"{LM}/{lang}.bin",
                                      unigrams=unigrams(lang, top), alpha=a, beta=b)
    decode_kwargs[lang] = BEAM_LANG.get(lang, {})
print("✓ prêt :", MODEL_NAME, "+", LM_SUBDIR, "(6 décodeurs, α/β par langue)")
print("(warnings pyctcdecode 'length > 1' / 'unigrams don't agree' = bénins)")

## 4. Données de test (Drive si présent, sinon téléchargement HF)

In [ ]:
import glob
TEST_CANDIDATES = [f"{BASE}/test", "/content/test_data"]
parquets = []
for td in TEST_CANDIDATES:
    parquets = sorted(glob.glob(f"{td}/**/*.parquet", recursive=True))
    if parquets:
        TEST_DIR = td; break
if not parquets:
    print("test absent localement -> téléchargement HF...")
    from huggingface_hub import snapshot_download
    TEST_DIR = "/content/test_data"
    snapshot_download("digitalumuganda/anv-test-data-nt", repo_type="dataset", local_dir=TEST_DIR)
    parquets = sorted(glob.glob(f"{TEST_DIR}/**/*.parquet", recursive=True))
print(f"{len(parquets)} parquets test dans {TEST_DIR}")
assert parquets, "aucun parquet test trouvé"

## 5. Inférence : courts en batch tronqué, longs en fenêtres v2 + recollage des logits

In [ ]:
import pandas as pd, time
from tqdm.auto import tqdm

MAX_SEC_BATCH = 160.0

# ---------- fenêtrage v2 ----------
def _rms(x, fl=400, hop=160):
    """Énergie RMS par trame de 25 ms (hop 10 ms)."""
    if len(x) < fl:
        return np.array([float(np.sqrt(np.mean(x ** 2) + 1e-12))]), hop
    nfr = 1 + (len(x) - fl) // hop
    idx = np.arange(nfr)[:, None] * hop + np.arange(fl)[None, :]
    fr = x[idx]
    return np.sqrt((fr ** 2).mean(axis=1) + 1e-12), hop

def fenetres_v2(arr, sr=16000):
    """[(début, fin)] en échantillons : coupe au minimum d'énergie dans les SEARCH_SEC
    dernières secondes de la fenêtre cible ; chevauchement OVER_SEC centré sur la coupe ;
    dernière fenêtre alignée sur la fin du clip. Longueur max : WIN + OVER/2 = 16s (< 20s)."""
    L = len(arr); w = int(WIN_SEC * sr); o = int(OVER_SEC * sr); s_s = int(SEARCH_SEC * sr)
    if L <= w: return [(0, L)]
    wins = []; cur = 0
    while L - cur > w:
        target = cur + w
        a = max(target - s_s, cur + o)
        seg = arr[a:min(target, L)]
        rms, hop = _rms(seg)
        cut = a + int(np.argmin(rms)) * hop + 200
        cut = min(max(cut, cur + o), L - 1)
        wins.append((cur, min(cut + o // 2, L)))
        cur = max(cut - o // 2, 0)
    wins.append((max(0, L - w), L))
    if len(wins) >= 2 and wins[-1][0] <= wins[-2][0]:
        wins[-2] = (wins[-2][0], L); wins.pop()
    return wins

def stitch(wins, lgs):
    """Assemble les logits des fenêtres en une seule ligne de temps.
    Chaque chevauchement est tranché en son MILIEU — qui tombe sur la coupe-silence,
    donc sur des frames quasi vides : chaque frame provient d'UNE seule fenêtre, avec
    >= OVER/2 de contexte de part et d'autre. Les indices sont relatifs à chaque fenêtre
    (spf propre), donc AUCUNE dérive d'alignement, quelle que soit la durée du clip."""
    if len(wins) == 1:
        return lgs[0].astype(np.float32)
    parts = []
    for i, ((s, e), lg) in enumerate(zip(wins, lgs)):
        nfr = lg.shape[0]
        if nfr == 0: continue
        spf = (e - s) / nfr
        lo = s if i == 0 else (wins[i - 1][1] + s) // 2
        hi = e if i == len(wins) - 1 else (e + wins[i + 1][0]) // 2
        a = max(0, int(round((lo - s) / spf)))
        b = min(nfr, int(round((hi - s) / spf)))
        if b > a:
            parts.append(lg[a:b].astype(np.float32))
    return np.concatenate(parts, axis=0)

# ---------- logits batchés + TRONQUÉS (anti-padding) ----------
def logits_batch(arrs):
    """Logits de chaque array (même ordre), batching dynamique par durée,
    chaque sortie TRONQUÉE à sa vraie longueur (les frames de padding
    n'ont jamais été entraînées : les décoder ajoute des mots parasites)."""
    order = sorted(range(len(arrs)), key=lambda k: len(arrs[k]))
    out = [None] * len(arrs); i = 0
    while i < len(order):
        j = i + 1
        while j < len(order) and (j - i + 1) * (len(arrs[order[j]]) / 16000.0) <= MAX_SEC_BATCH:
            j += 1
        idxs = order[i:j]
        feats = processor.feature_extractor([arrs[k] for k in idxs], sampling_rate=16000,
                                            return_tensors="pt", padding=True)
        am = feats.get("attention_mask")
        with torch.no_grad():
            lg = model(**{k: v.to(device) for k, v in feats.items()}).logits.cpu().numpy()
        T_out = lg.shape[1]
        if am is not None:
            fr = am.sum(-1).numpy().astype(float); fmax = float(am.shape[1])
        else:
            fr = np.array([len(arrs[k]) for k in idxs], dtype=float); fmax = fr.max()
        for bi, k in enumerate(idxs):
            n_true = max(1, int(round(T_out * fr[bi] / fmax)))
            out[k] = lg[bi, :min(n_true, T_out)]
        i = j
    return out

# ---------- boucle principale (reprise auto) ----------
done = {}
if os.path.exists(PARTIEL):
    dfp = pd.read_csv(PARTIEL)
    done = dict(zip(dfp["id"].astype(str), zip(dfp["language"], dfp["transcription"])))
    print("reprise :", len(done), "déjà transcrits")

rows = [{"id": k, "language": v[0], "transcription": v[1]} for k, v in done.items()]
t0 = time.time()

for pi, pq in enumerate(tqdm(parquets, desc="Inférence", unit="pq")):
    df = pd.read_parquet(pq)
    buckets = {}   # lang -> [(rid, arr)]
    for _, r in df.iterrows():
        rid = str(r["id"])
        if rid in done: continue
        lang = r.get("language")
        b = r["audio"].get("bytes") if isinstance(r["audio"], dict) else r["audio"]
        try:
            arr = decode_robuste(b)
        except Exception:
            rows.append({"id": rid, "language": lang, "transcription": "_"})
            continue
        buckets.setdefault(lang, []).append((rid, arr))

    for lang, its in buckets.items():
        dec = decoders.get(lang) or next(iter(decoders.values()))
        kw  = decode_kwargs.get(lang, {})
        seg_arrs, seg_map = [], []
        for rid, arr in its:
            if len(arr) / 16000.0 <= SEUIL_SEC:
                seg_map.append((rid, None)); seg_arrs.append(arr)
            else:
                ws = fenetres_v2(arr)
                for (s, e) in ws:
                    seg_map.append((rid, (s, e))); seg_arrs.append(arr[s:e])
        lgs = logits_batch(seg_arrs)
        par_rid = {}
        for k, (rid, w) in enumerate(seg_map):
            par_rid.setdefault(rid, []).append((w, lgs[k]))
        for rid, segs in par_rid.items():
            try:
                if len(segs) == 1 and segs[0][0] is None:
                    txt = dec.decode(segs[0][1], **kw)
                else:
                    segs.sort(key=lambda x: x[0][0])
                    st = stitch([w for w, _ in segs], [lg for _, lg in segs])
                    txt = dec.decode(st, **kw)
            except Exception:
                txt = "_"
            rows.append({"id": rid, "language": lang, "transcription": txt})

    if (pi + 1) % 10 == 0 or pi == len(parquets) - 1:
        pd.DataFrame(rows).to_csv(PARTIEL, index=False)
        print(f"parquet {pi + 1}/{len(parquets)} | {len(rows)} transcrits | {(time.time() - t0) / 60:.0f} min")

print("✓ inférence fenêtres v2 terminée :", len(rows))

## 6. Assemblage final + asserts + CSV

In [ ]:
import pandas as pd
sub = pd.DataFrame(rows)
sub["transcription"] = (sub["transcription"].astype(str)
                        .str.replace(r"[\r\n]+", " ", regex=True).str.strip())
sub.loc[sub["transcription"].isin(["", "nan", "None"]), "transcription"] = "_"
sub["id"] = sub["id"].astype(str)
sub = sub.drop_duplicates(subset="id", keep="first")[["id", "language", "transcription"]]

assert list(sub.columns) == ["id", "language", "transcription"]
assert len(sub) == 41733, f"attendu 41733, obtenu {len(sub)} — vérifie les parquets/reprise"
assert sub["id"].is_unique
assert sub["language"].notna().all()
assert (sub["transcription"].str.len() > 0).all()

sub.to_csv(FINAL, index=False)
print("✅ soumission écrite ->", FINAL)
print(sub["language"].value_counts())
sub.head(8)

## 7. Après la soumission

- Compare à **0.40283** (v9-2 direct) et à ta soumission fenêtres v1. Note le delta réel
  vs le delta projeté (§7 du notebook P0) : c'est ta nouvelle calibration.
- **Un seul changement par soumission** si tu veux attribuer les gains : d'abord
  fenêtres v2 seule (AB global), puis les (α, β) par langue si la grille les justifie.
- Si ça gagne : mets à jour README / MODEL_CARD / RAPPORT (le fenêtrage est un
  post-traitement d'inférence pur : aucun réentraînement, RTF +~15 % sur les clips
  longs — re-mesure le RTF max pour HARDWARE_VALIDATION, largement ≤ 2×).
- Le dépôt doit inclure ce notebook (règle 6b) à la place / à côté de la v1.